In [1]:
import numpy as np
from numpy.polynomial.legendre import leggauss


def bates_call(S, K, T, r, v0, theta, kappa, xi, rho, lam, jump_mean, jump_std, q=0.0):
    """European call under the Bates (1996) model (Heston stochastic vol + lognormal jumps).
    Everything annual and decimal. v0, theta are VARIANCES (0.16 = 40% vol). T in years (> 0).
    K can be a number or an array. Put = call - S*exp(-q*T) + K*exp(-r*T).
    FX: r = domestic rate, q = foreign rate.  Oil: S = futures price, q = r."""
    K = np.atleast_1d(np.asarray(K, float))
    m, sj = jump_mean, jump_std
    xi = max(xi, 1e-3)
    mu = np.log1p(m) - 0.5 * sj ** 2
    fwd, disc = S * np.exp(-q * T), K * np.exp(-r * T)
    lnK = np.log(K)[:, None]
    x, w0 = leggauss(64)

    def cf(u):  # characteristic function of ln(S_T), 
        b = kappa - 1j * rho * xi * u
        d = np.sqrt(b * b + xi ** 2 * (1j * u + u * u))
        g = (b - d) / (b + d)
        e = np.exp(-d * T)
        C = kappa * theta / xi ** 2 * ((b - d) * T - 2 * np.log((1 - g * e) / (1 - g)))
        D = (b - d) / xi ** 2 * (1 - e) / (1 - g * e)
        jumps = lam * T * (np.exp(1j * u * mu - 0.5 * u * u * sj ** 2) - 1 - 1j * u * m)
        return np.exp(1j * u * (np.log(S) + (r - q) * T) + C + D * v0 + jumps)

    def integrate(a, b):  # integral over u in (a, b) of the P1 and P2 integrands
        n = max(int(np.ceil((b - a) / 40.0)), 1)
        h = (b - a) / n
        u = (a + (np.arange(n) * h)[:, None] + 0.5 * h * (x + 1)).ravel() + 1e-10
        w = np.tile(0.5 * h * w0, n)
        U = u[None, :]
        ph = np.exp(-1j * U * lnK)
        i2 = (w * np.real(ph * cf(U) / (1j * U))).sum(1)
        i1 = (w * np.real(ph * cf(U - 1j) / (S * np.exp((r - q) * T)) / (1j * U))).sum(1)
        return i1 / np.pi, i2 / np.pi

    # start with a range sized by sigma*sqrt(T), then keep doubling until the tail is negligible
    total_var = max(v0, theta, 1e-8) + lam * (mu ** 2 + sj ** 2)
    hi = float(np.clip(8.0 / np.sqrt(total_var * T), 200.0, 1e5))
    I1, I2 = integrate(0.0, hi)
    while hi < 1e6:
        t1, t2 = integrate(hi, 2 * hi)
        I1, I2, hi = I1 + t1, I2 + t2, 2 * hi
        if np.max(np.abs(fwd * t1 - disc * t2)) < 1e-9 * S:
            break

    price = fwd * (0.5 + I1) - disc * (0.5 + I2)
    return price if price.size > 1 else float(price[0])
